# Fig. 3 and Extended Data Figs 3-4: s captures the plaque-associated microglial niche (STARmap + CODEX)

Mouse AD STARmap (2,766 genes; 13-month control vs disease) and human AD CODEX whole slices (34 proteins;
4 AD vs 3 control donors). Amyloid plaques are never given to the model; all readouts use disease-arm cells.
Held-out prediction of plaque proximity is stronger from s than from z or scVI near plaques; over 3 training seeds
rho(s) > rho(z) in 35/36 CODEX and 13/15 STARmap comparisons. The salient composition shift D([z,s]) - D([z,0])
reproduces the observed microglia enrichment toward plaques that the background prediction D([z,0]) misses, and the
s microglia axis separates DAM (Cst7, Apoe) from homeostatic (P2ry12) microglia. The plaque axis is conserved
across species (20 microglial protein-ortholog pairs: rho = +0.47 gradient, +0.54 s loading). Controls: spatially
shifted plaques, a control-vs-control arm swap (Ctl-3 relabelled as disease) and s-beyond-z tests (ED Fig. 3).

In [ ]:
import sys, pathlib; sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), pathlib.Path.cwd() / 'notebooks'] if (p / 'repro.py').exists())))
from repro import run, require, show, fetch, fetch_fitted, fetch_baselines, STAGES, H, WK, PY, PY_VIO, GPU
# Default: download inputs + the paper's fitted Haruka outputs from Hugging Face, then analyses -> figure.
# The Haruka fit step reuses the unpacked fit; HARUKA_REFIT=1 (env) refits from scratch. Other Haruka fits (ablations,
# extra seeds) run. Baseline methods are not refitted: their saved results are downloaded (RUN_BASELINE=1 refits them).
# Redraw only from saved outputs: STAGES.update(download=False, fit=False, analysis=False)  (or env HARUKA_FIGURE_ONLY=1)
# Rebuild the inputs from the raw public releases instead of Hugging Face: STAGES.update(data=True)
print(H, STAGES)

## 0 Preprocessed data (Hugging Face)

The ready-to-fit inputs are read directly from [C1vy/haruka_v2_data](https://huggingface.co/datasets/C1vy/haruka_v2_data) (cached in `hf_data/`) and linked to the paths the scripts read. The data stage below rebuilds the same files from the raw releases and is off by default.

In [ ]:
fetch('03_fig3_ad_plaque/ad_starmap.h5ad', H / 'bench_real_discovery/data/ad.h5ad')
fetch('03_fig3_ad_plaque/codex_wholeslice_all7_no_oligo.h5ad', H / 'bench_codex_cns_2026/wholeslice/codex_wholeslice_all7_no_oligo.h5ad')
fetch('03_fig3_ad_plaque/codex_wholeslice_ctl3VsCtl14.h5ad', H / 'bench_codex_cns_2026/wholeslice/codex_wholeslice_ctl3VsCtl14.h5ad')

The paper's default Haruka fit, published as fitted AnnData (input + z, s, predicted neighbourhoods; see the dataset card). It is unpacked into the fit folder, so the fit step below reuses it. Set `HARUKA_REFIT=1` before launching to refit instead.

In [ ]:
fetch_fitted('fitted/03_fig3_ad_plaque/ad_haruka_fitted.h5ad', H / 'results_real_compgated_lambda10_samplebatch_sepheads/ad/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0')
fetch_fitted('fitted/03_fig3_ad_plaque/codex_wholeslice_all7_no_oligo_haruka_fitted.h5ad', H / 'results_codex_wholeslice/codex_wholeslice_all7_no_oligo/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0')
fetch_fitted('fitted/03_fig3_ad_plaque/codex_wholeslice_ctl3VsCtl14_haruka_fitted.h5ad', H / 'results_codex_wholeslice/codex_wholeslice_ctl3VsCtl14/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0')

Saved results of the comparison methods (scVI, ContrastiveVI, expression, CINEMA-OT), so only the Haruka results are recomputed. `RUN_BASELINE=1` refits the baselines instead.

In [ ]:
fetch_baselines('baselines/03_fig3_ad_plaque_baselines.tar.gz')

## 1 Data

In [ ]:
run(f'{PY} bench_real_discovery/prepare_studies.py ad', 'data')                     # STARmap -> bench_real_discovery/data/ad.h5ad
W = H / 'bench_codex_cns_2026/wholeslice'
run(f'{PY} build_wholeslice.py', 'data', cwd=W)                                   # CODEX all7_no_oligo (+ full3)
run(f'{PY} build_ctl3_vs_ctl14.py', 'data', cwd=W)                               # arm-swap control design

## 2 Fits (default model; normal likelihood on pretransformed input; ~12 min STARmap, ~3.5 min per CODEX design) + seed checks

In [ ]:
run(f'{PY} bench_kras_xenium/run_haruka_separate_batch_heads.py --data bench_real_discovery/data/ad.h5ad --batch_size 4096 --sample_key slice_id --batch_key slice_id --condition_key condition --target_label disease --cell_type_key cell_type --gene_likelihood normal --neighbor_input_mode pretransformed --assay_type sequencing --n_top_genes 0 --out_root results_real_compgated_lambda10_samplebatch_sepheads --seed 0 --max_epochs 150 --n_latent 32 --n_hidden 256 --n_layers 1 --decoder_layers 2 --k_niche 20 --neighbor_embedding_dim 16 --composition_loss_weight 1 --neighbor_mean_loss_weight 1 --neighbor_expression_mode composition_gated --spatial_weight 100 --background_wasserstein_penalty 0 --zs_independence_penalty 10 --decoder_batch_dim 8 --batch_injection_mode decoder_only --augmentation_mode sparse --num_workers 4 --direct_only', 'fit')
run(f'{PY} bench_kras_xenium/rebuild_condition_only_salient_cc.py --data bench_real_discovery/data/ad.h5ad --input-dir results_real_compgated_lambda10_samplebatch_sepheads/ad/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0 --output-dir results_real_compgated_lambda10_samplebatch_sepheads/ad/haruka_z_s_cellcharter_cc3_condition_only --condition-key condition --condition-value disease --slice-key slice_id', 'analysis')
for v in ['all7_no_oligo', 'ctl3VsCtl14']:
    run(f'bash run_wholeslice_fit.sh {v} {GPU}', 'fit', cwd=W)
for ds in ['starmap', 'codex']:
    for seed in [1, 2]:
        run(f'bash results_seedcheck/run_seed.sh {ds} {seed} {GPU}', 'fit')

## 3 Plaque distances (model-independent) and CODEX radial / null analyses

In [ ]:
# STARmap plaque distances: ad_analysis/plot_plaque_distance_spatial.py -> final_results_nb/starmap_ad/biology/plaque_distance/starmap_plaque_distance_cells.csv.gz
require(H / 'final_results_nb/starmap_ad/biology/plaque_distance/starmap_plaque_distance_cells.csv.gz')
PS = W / 'plaque_states'; SEED = H / 'results_seedcheck/codex/codex_wholeslice_all7_no_oligo/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed'
run(f'{PY} build_plaque_niches.py', 'analysis', cwd=PS)
run(f'{PY} radial_generic.py all7_no_oligo && {PY} radial_generic.py all7_no_oligo all && {PY} radial_generic.py ctl3VsCtl14', 'analysis', cwd=PS)
for k in [1, 2]:
    run(f'{PY} radial_generic.py all7_no_oligo', 'analysis', cwd=PS, env={'MODEL_DIR': f'{SEED}{k}', 'TAG': f'_seed{k}'})
run(f'{PY} radial_null.py all7_no_oligo', 'analysis', cwd=PS)

## 4 STARmap shifted-plaque null (with scVI), s beyond z, counterfactual proportions, cross-species markers, domains

In [ ]:
SR = H / 'bench_real_discovery/starmap_plaque_radial'
SSEED = H / 'results_seedcheck/starmap/ad/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed'
run(f'{PY} fit_scvi.py', 'baseline', cwd=SR)   # scVI baseline (off: latent downloaded in section 0)
run(f'{PY} starmap_null.py', 'analysis', cwd=SR)
for k in [1, 2]:
    run(f'{PY} starmap_null.py', 'analysis', cwd=SR, env={'MODEL_DIR': f'{SSEED}{k}', 'TAG': f'_seed{k}'})
run(f'{PY} s_beyond_z.py starmap && {PY} s_beyond_z.py codex', 'analysis', cwd=SR)
run(f'{PY} bench_real_discovery/cc_shift_info/check_cf_proportion.py starmap && {PY} bench_real_discovery/cc_shift_info/check_cf_proportion.py codex', 'analysis')
run(f'{PY} marker_axes.py starmap && {PY} marker_axes.py codex && {PY} compare_species.py', 'analysis', cwd=PS / 'cross_species')
run(f'{PY} bench_real_discovery/analyze_domain_celltype_enrichment.py --data bench_real_discovery/data/ad.h5ad --cc-dir results_real_compgated_lambda10_samplebatch_sepheads/ad/haruka_z_s_cellcharter_cc3_condition_only --out-dir final_results_nb/starmap_ad/biology/domain_annotation --sample-key slice_id --cell-type-key cell_type --seed 0', 'analysis')

## 5 Panels and pages (letters from `fig3_layout.py`)

In [ ]:
F3 = H / 'figures_paper/fig3'
run(f'{PY} haruka_domain_plaque_stats.py', 'analysis', cwd=F3)
for s in ['make_fig3_haruka_blocks.py', 'make_fig3_plaque.py', 'make_fig3_cf_proportion.py', 'make_fig3_markers.py', 'make_fig3_full.py']:
    run(f'{PY} {s}', 'figure', cwd=F3)

## Figure

In [ ]:
show(H / 'figures_paper/fig3/final/Fig3_full.png')

In [ ]:
show(H / 'figures_paper/fig3/final/EDFig3_full.png')

In [ ]:
show(H / 'figures_paper/fig3/final/EDFig4_full.png')